# Kiểm tra `fps_map.json` có khớp với video thật không

Bảng fps của BTC là thứ **cả pipeline dựa vào** để đổi giây sang frame. Nếu một dòng
trong bảng sai, mọi frame của video đó bị đánh số lệch — và không có gì báo cho tới lúc
chấm điểm.

Notebook này đọc từng file `.mp4`, dò fps bằng ba cách độc lập, rồi so với bảng.

| | |
|---|---|
| **Đầu vào** | `/content/drive/MyDrive/aic25/AIC_Video/**/*.mp4` · `aic26/fps_map.json` |
| **Đầu ra** | `aic26/fps_check_result.json` — kết quả từng video |
| **Cần GPU** | không |
| **Thời gian** | ~0,3–1 giây mỗi video · 1478 video ≈ **10–25 phút** |

## Ba cách dò, vì sao cần cả ba

| Cách | ffprobe trả gì | Đọc thế nào |
|---|---|---|
| `r_frame_rate` | nhịp khung **danh nghĩa** khai trong container | phân số chính xác, vd `30000/1001` |
| `avg_frame_rate` | tổng frame ÷ thời lượng | phản ánh thực tế, lệch nếu video có khung rơi |
| `nb_frames / duration` | đếm thật ÷ giây thật | chậm hơn nhưng khó sai nhất |

Ba số này **khớp nhau ở video bình thường**. Chỗ nào chúng lệch nhau là chỗ đáng ngờ,
kể cả khi một trong ba trùng với bảng.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 00 – Cấu hình
# ═══════════════════════════════════════════════════════════════════════════════
# %%

# Kiểm bao nhiêu video. 0 = kiểm hết (1478 video, khoảng 10–25 phút).
# Để 30 chạy thử trước cho nhanh.
MAX_VIDEOS = 30

# Chỉ kiểm các nhóm này. Để trống = mọi nhóm.
# Ví dụ ["K01", "K02"] để soi riêng batch 2.
BATCHES = []

# Sai lệch fps được coi là khớp. 0.01 đủ để 29.970029... khớp với 29.97 trong bảng,
# nhưng vẫn bắt được 29.97 vs 30.0 (lệch 0.03).
MATCH_TOL = 0.01

VIDEO_ROOT   = "/content/drive/MyDrive/aic25/AIC_Video"
SHARED_DIR   = "/content/drive/MyDrive/aic26"
FPS_MAP_PATH = f"{SHARED_DIR}/fps_map.json"
RESULT_PATH = f"{SHARED_DIR}/fps_check_result.json"

print(f"[Config] checking {MAX_VIDEOS or 'ALL'} videos"
      + (f" · batches {BATCHES}" if BATCHES else " · all batches"))
print(f"[Config] match tolerance: {MATCH_TOL}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 01 – Mount Drive + nạp bảng fps
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import os, json, glob, subprocess as _sp
from pathlib import Path
from collections import Counter, defaultdict

DRIVE_MOUNT = "/content/drive"
if os.path.ismount(DRIVE_MOUNT):
    print("[Drive] Already mounted.")
else:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
    print("[Drive] Mounted.")

try:
    from tqdm.auto import tqdm
except ImportError:
    _sp.check_call(["pip", "install", "-q", "tqdm"])
    from tqdm.auto import tqdm

if not os.path.exists(FPS_MAP_PATH):
    raise FileNotFoundError(
        f"Thiếu {FPS_MAP_PATH}\n"
        "  → Chép data_raw/fps_map.json trong repo lên đúng đường dẫn đó."
    )

with open(FPS_MAP_PATH) as f:
    FPS_MAP = {k: float(v) for k, v in json.load(f).items()}

_by_batch = Counter(k.split("_")[0] for k in FPS_MAP)
print(f"\n[Table] {len(FPS_MAP)} videos · {len(_by_batch)} batches")
print("[Table] " + " · ".join(f"{k}:{v}" for k, v in sorted(_by_batch.items())))
print("[Table] fps: " + " · ".join(f"{k}={v}" for k, v in
                                  sorted(Counter(FPS_MAP.values()).items())))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 02 – Hàm dò fps bằng ffprobe
# ═══════════════════════════════════════════════════════════════════════════════
# %%

def _fraction(s):
    """ffprobe trả nhịp khung dạng phân số, vd '30000/1001'. Giữ nguyên phân số
    rồi mới chia — đọc thẳng float sẽ mất chính xác ở 29.97."""
    try:
        if s in (None, "", "0/0", "N/A"):
            return None
        if "/" in s:
            n, d = s.split("/")
            return float(n) / float(d) if float(d) else None
        return float(s)
    except Exception:
        return None


def probe_fps(path: str) -> dict:
    """Trả ba số fps độc lập + thời lượng + số frame.

    r_frame_rate     nhịp danh nghĩa khai trong container
    avg_frame_rate   tổng frame ÷ thời lượng, do ffprobe tự tính
    nb/duration      đếm thật chia giây thật, chỉ có khi container ghi nb_frames
    """
    ra = {"r": None, "avg": None, "nb_div_dur": None,
          "duration": None, "nb_frames": None, "error": None}
    try:
        r = _sp.run(
            ["ffprobe", "-v", "error", "-select_streams", "v:0",
             "-show_entries",
             "stream=r_frame_rate,avg_frame_rate,nb_frames,duration:format=duration",
             "-of", "json", path],
            capture_output=True, text=True, timeout=60)
        info = json.loads(r.stdout or "{}")
        st   = (info.get("streams") or [{}])[0]
        fm   = info.get("format") or {}

        ra["r"]   = _fraction(st.get("r_frame_rate"))
        ra["avg"] = _fraction(st.get("avg_frame_rate"))

        dur = st.get("duration") or fm.get("duration")
        ra["duration"] = float(dur) if dur not in (None, "N/A") else None

        nb = st.get("nb_frames")
        if nb not in (None, "N/A"):
            ra["nb_frames"] = int(nb)
            if ra["duration"]:
                ra["nb_div_dur"] = ra["nb_frames"] / ra["duration"]
    except Exception as e:
        ra["error"] = str(e)[:120]
    return ra


print("[Func] probe_fps() ready.")
print("[Probe] ffprobe available:",
      _sp.run(["ffprobe", "-version"], capture_output=True,
              text=True).stdout.split("\n")[0] or "NOT FOUND")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 03 – Quét video và so với bảng
# ═══════════════════════════════════════════════════════════════════════════════
# %%

_all_sorted = sorted(set(
    glob.glob(os.path.join(VIDEO_ROOT, "**", "*.mp4"), recursive=True) +
    glob.glob(os.path.join(VIDEO_ROOT, "*.mp4"))
))
if not _all_sorted:
    raise FileNotFoundError(f"No videos found in {VIDEO_ROOT}")

print(f"[Videos] {len(_all_sorted)} .mp4 files on Drive")

# Ba tập lệch nhau — báo ra trước khi kiểm
_video_names = {Path(v).stem for v in _all_sorted}
_missing_from_table = sorted(_video_names - set(FPS_MAP))       # có video, bảng không có
_missing_from_drive = sorted(set(FPS_MAP) - _video_names)      # bảng có, không có video

if _missing_from_table:
    print(f"\n[!] {len(_missing_from_table)} videos NOT in the fps table - cannot be cut:")
    print(f"    {_missing_from_table[:12]}{' …' if len(_missing_from_table) > 12 else ''}")
if _missing_from_drive:
    print(f"\n[i] {len(_missing_from_drive)} table entries with no video on Drive yet:")
    print(f"    {_missing_from_drive[:12]}{' …' if len(_missing_from_drive) > 12 else ''}")

# Lọc theo cấu hình
to_check = [v for v in _all_sorted if Path(v).stem in FPS_MAP]
if BATCHES:
    to_check = [v for v in to_check if Path(v).stem.split("_")[0] in set(BATCHES)]
if MAX_VIDEOS:
    to_check = to_check[:MAX_VIDEOS]

print(f"\n[Check] {len(to_check)} videos\n")

RESULTS = []
for p in tqdm(to_check, desc="ffprobe"):
    ten   = Path(p).stem
    table_fps = FPS_MAP[ten]
    d     = probe_fps(p)

    # Chọn số đại diện để so: ưu tiên r_frame_rate (đây là số container khai và
    # cũng là số ffprobe trả về mặc định — chính là số mà một pipeline dò fps sẽ
    # nhận được nếu không có bảng).
    probed = d["r"] or d["avg"] or d["nb_div_dur"]

    delta     = abs(probed - table_fps) if probed else None
    match     = (delta is not None and delta <= MATCH_TOL)
    # Số frame bị lệch ở CUỐI video — đây mới là con số có ý nghĩa thực tế
    drift_end = (d["duration"] * delta) if (delta and d["duration"]) else None
    # Ba số dò có tự mâu thuẫn nhau không
    _probes = [x for x in (d["r"], d["avg"], d["nb_div_dur"]) if x]
    probes_off = (max(_probes) - min(_probes) > MATCH_TOL) if len(_probes) > 1 else False

    RESULTS.append({
        "video": ten, "table": table_fps, "probed": probed,
        "r": d["r"], "avg": d["avg"], "nb_div_dur": d["nb_div_dur"],
        "duration": d["duration"], "nb_frames": d["nb_frames"],
        "delta": delta, "match": match, "frame_drift": drift_end,
        "probes_disagree": probes_off, "error": d["error"],
    })

print(f"\n[Done] {len(RESULTS)} videos")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 04 – Báo cáo
# ═══════════════════════════════════════════════════════════════════════════════
# %%

matched  = [r for r in RESULTS if r["khop"]]
mismatch = [r for r in RESULTS if not r["khop"] and r["probed"]]
failed   = [r for r in RESULTS if r["loi"] or not r["probed"]]
disagree = [r for r in RESULTS if r["ba_so_mau_thuan"]]

print("=" * 78)
print(f"  fps_map.json CHECK - {len(RESULTS)} videos")
print("=" * 78)
print(f"  Match table       : {len(matched):5d}  ({len(matched)/max(len(RESULTS),1)*100:.1f}%)")
print(f"  MISMATCH          : {len(mismatch):5d}")
print(f"  Unreadable        : {len(failed):5d}")
print(f"  Probes disagree   : {len(disagree):5d}")
print("=" * 78)

if mismatch:
    print()
    print("  MISMATCHED VIDEOS - sorted by frame drift at end of video")
    print(f"  {'video':<14}{'table':>8}{'probed':>10}{'delta':>9}"
          f"{'secs':>8}{'frame drift':>17}")
    print("  " + "-" * 74)
    for r in sorted(mismatch, key=lambda x: -(x["frame_drift"] or 0)):
        print(f"  {r['video']:<14}{r['table']:>8.3f}{r['probed']:>10.4f}"
              f"{r['delta']:>9.4f}{(r['duration'] or 0):>8.0f}"
              f"{(r['lech_frame_cuoi'] or 0):>17.1f}")
    print()
    print("  'frame drift' = frames mis-numbered at the END of the video.")
    print("  Even under 10 frames, the TRAKE window (<10 frames) is at risk.")

if disagree:
    print()
    print("  PROBES DISAGREE - suspicious even when the table matches")
    print(f"  {'video':<14}{'r_frame':>10}{'avg_frame':>12}{'nb/dur':>10}{'table':>9}")
    print("  " + "-" * 60)
    for r in disagree[:20]:
        print(f"  {r['video']:<14}{(r['r'] or 0):>10.4f}{(r['avg'] or 0):>12.4f}"
              f"{(r['nb_div_dur'] or 0):>10.4f}{r['table']:>9.3f}")
    print("  Usually videos with dropped frames or a variable frame rate.")

if failed:
    print()
    print("  UNREADABLE")
    for r in failed[:20]:
        print(f"    {r['video']}: {r['error'] or 'ffprobe returned nothing'}")

# Đối chiếu theo cặp giá trị: bảng ghi X mà video là Y
print()
print("  TABLE  ->  PROBED   (rounded to 2 decimals)")
pairs = Counter((r["table"], round(r["probed"], 2)) for r in RESULTS if r["probed"])
for (b, t), n in sorted(pairs.items(), key=lambda x: -x[1]):
    dau = "✓" if abs(b - t) <= MATCH_TOL else "✗"
    print(f"    {dau}  {b:>7} → {t:<7}  {n:5d} video")

with open(RESULT_PATH, "w") as f:
    json.dump(RESULTS, f, indent=1, ensure_ascii=False)
print(f"\n[Saved] {RESULT_PATH}")

---

## Kiểm tra thêm — bảng có tự nhất quán không

Ô dưới **không cần video**, chỉ cần thư mục `map-keyframes` gốc của BTC.

Nó kiểm một điều khác hẳn phần trên: trong chính file CSV của BTC, `frame_idx` được
tính từ `pts_time × fps` bằng **phép làm tròn nào**. Ô này thử cả bốn — `round`,
`floor`, `ceil`, `int` — và chỉ ra phép nào khớp 100%.

Quan trọng vì nếu dùng sai phép, mọi chỗ đổi giây↔frame trong pipeline lệch 1 frame —
mà cửa sổ đáp án TRAKE chỉ rộng dưới 10 frame.

> Đo trước trên bộ dữ liệu hiện có: **`floor` khớp 100%**, `round` chỉ **83,5%**.
> Ô này để bạn tự xác nhận lại trên đúng dữ liệu đang dùng.

Bỏ qua ô này nếu chưa đẩy `map-keyframes` lên Drive.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 05 – (tuỳ chọn) Bảng có tự nhất quán không: frame_idx = pts_time × fps
# ═══════════════════════════════════════════════════════════════════════════════
# %%

import csv

MAP_DIRS = [f"{SHARED_DIR}/map-keyframes-aic25-b1",
            f"{SHARED_DIR}/map-keyframes-aic25-b2"]

_files = []
for d in MAP_DIRS:
    _files += sorted(glob.glob(os.path.join(d, "**", "*.csv"), recursive=True))

if not _files:
    print("[Skip] No map-keyframes folder on Drive.")
    print(f"         Needs: {MAP_DIRS[0]}")
else:
    import math

    print(f"[CSV] {len(_files)} files\n")

    # Thử BỐN phép làm tròn thay vì giả định một phép. Phép nào khớp 100% chính
    # là phép BTC dùng — và cũng là phép pipeline phải dùng khi đổi giây → frame.
    ROUNDING = {
        "round(t×fps)": lambda x: round(x),
        "floor(t×fps)": math.floor,
        "ceil(t×fps)":  math.ceil,
        "int(t×fps)":   int,
    }
    hits   = {k: 0 for k in ROUNDING}
    n_rows = 0
    examples  = []
    fps_in_file = {}

    for p in tqdm(_files, desc="đọc csv"):
        ten = Path(p).stem
        try:
            with open(p, encoding="utf-8-sig", newline="") as f:
                rows = list(csv.DictReader(f))
        except Exception as e:
            print(f"  [error] {ten}: {e}")
            continue

        values = {float(r["fps"]) for r in rows if r.get("fps")}
        fps_in_file[ten] = sorted(values)

        for r in rows:
            try:
                t, fps, idx = float(r["pts_time"]), float(r["fps"]), int(r["frame_idx"])
            except Exception:
                continue
            n_rows += 1
            x = t * fps
            for k, f_ in ROUNDING.items():
                if f_(x) == idx:
                    hits[k] += 1
            if round(x) != idx and len(examples) < 20:
                examples.append((ten, t, fps, idx, round(x, 3)))

    print(f"\n[Consistency] {n_rows:,} rows\n")
    print(f"  {'formula':<16}{'hits':>12}{'ratio':>10}")
    print("  " + "-" * 38)
    for k, v in sorted(hits.items(), key=lambda x: -x[1]):
        dau = "✓" if v == n_rows else " "
        print(f"  {dau} {k:<14}{v:>12,d}{v/max(n_rows,1)*100:>9.2f}%")

    _winner = [k for k, v in hits.items() if v == n_rows]
    print()
    if _winner:
        print(f"  -> Organiser numbers frames as: frame_idx = {_winner[0]}")
        if "round(t×fps)" not in _winner:
            print()
            print("  WARN round() does NOT match. Every second -> frame conversion in the")
            print("    pipeline must use the formula above, not round(). One frame sounds")
            print("    small, but the TRAKE window is under 10 frames wide.")
            print("\n  Rows where round() gives a different result:")
            print(f"  {'video':<14}{'pts_time':>10}{'fps':>8}{'frame_idx':>11}{'t×fps':>10}")
            print("  " + "-" * 54)
            for v in examples[:10]:
                print(f"  {v[0]:<14}{v[1]:>10.4f}{v[2]:>8.2f}{v[3]:>11d}{v[4]:>10.3f}")
    else:
        print("  [!] NO formula matches 100% - the organiser numbering is more complex")
        print("      than a plain multiplication. Review before trusting any")
        print("      second <-> frame conversion in the pipeline.")

    multi_fps = {k: v for k, v in fps_in_file.items() if len(v) > 1}
    print()
    if multi_fps:
        print(f"  [!] {len(multi_fps)} videos have MULTIPLE fps values in one file:")
        for k, v in list(multi_fps.items())[:10]:
            print(f"        {k}: {v}")
    else:
        print("  OK each video has exactly one fps value in its file.")

---

## Đọc kết quả thế nào

**`Khớp bảng` gần 100%** → bảng dùng được, cứ chạy cắt frame.

**Có video `LỆCH`** → xem cột `frame lệch cuối`. Đó là số frame bị đánh số sai ở cuối
video, tính bằng `thời lượng × độ lệch fps`. Dưới 10 frame đã đủ làm trượt cửa sổ TRAKE;
vài trăm frame thì mọi câu hỏi về video đó đều sai.

Gặp lệch thì **tin bảng của BTC, không tin ffprobe** — BTC dùng chính bảng đó để đánh số
đáp án, nên dù ffprobe đúng về mặt kỹ thuật thì nộp theo ffprobe vẫn sai điểm. Ghi lại
danh sách lệch để biết video nào rủi ro, đừng sửa bảng theo ffprobe.

**`Ba số dò mâu thuẫn`** → video đó có khung rơi hoặc nhịp khung biến thiên. Bảng vẫn
đúng, nhưng `cap.set(CAP_PROP_POS_FRAMES)` lúc cắt keyframe có thể nhảy không chính xác.
Đáng đánh dấu để kiểm bằng mắt vài frame.

**`Không đọc được`** → file hỏng hoặc chưa tải xong từ Drive. Chạy lại riêng nhóm đó.